<a href="https://colab.research.google.com/github/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/colab/05_distance_and_contact_maps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the Google Colab version of this notebook. A copy with all its output is [in the SOURSOP repository](https://github.com/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/05_distance_and_contact_maps.ipynb).

# 5. Distance and contact maps

Global measures like Rg summarise a whole chain in one number. Inter-residue distance and contact maps show *where* two ensembles differ. Here we compare two alpha-synuclein ensembles from different methods (IDPForge and AF-CALVADOS; see [notebook 3](https://colab.research.google.com/github/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/colab/03_one_protein_many_methods.ipynb)), both with the same 140 residues and 100 conformers.

In [ ]:
# Colab does not come with SOURSOP, so install it first. This uses uv, which
# is much faster than pip (and is itself installed first if it is missing);
# --system installs into Colab's own Python rather than a virtual environment.
# The PED module is new in SOURSOP 2.0.8; if that version is not on PyPI yet,
# this installs the latest version from GitHub instead.
!command -v uv >/dev/null || pip install -q uv
!uv pip install --system -q "soursop>=2.0.8" 2>/dev/null || uv pip install --system -q "git+https://github.com/holehouse-lab/soursop.git"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from soursop import ped

# Figure style: Arial throughout, text kept editable in Illustrator, no grid
# lines, and the xmgrace colour order (black, red, blue, green, purple).
# Colab does not have Arial, so there figures fall back to Liberation Sans
# (metrically identical to Arial) or matplotlib's default, DejaVu Sans
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.grid": False,
    "axes.prop_cycle": plt.cycler(color=["black", "red", "blue", "green", "purple"]),
})

# Downloaded ensembles are kept here and reused, so each is only fetched once
# per session (Colab deletes them when the session ends)
CACHE = "ped_cache"

In [ ]:
ensembles = {"IDPForge": "PED17052e001", "AF-CALVADOS": "PED20241e001"}
proteins = {k: ped.load_ensemble(v, cache_dir=CACHE).proteinTrajectoryList[0] for k, v in ensembles.items()}
for k, p in proteins.items():
    print(f"{k:12s} {p.n_frames} conformers, {p.n_residues} residues")

## Mean distance maps

`get_distance_map()` returns the mean (or RMS) C-alpha-C-alpha distance for every residue pair, as an upper-triangular matrix, together with the standard deviation. We mirror it to a full matrix for plotting.

In [ ]:
def full(upper):
    """Mirror an upper-triangular map into a full symmetric matrix."""
    return upper + upper.T

maps = {k: full(p.get_distance_map(mode="CA", verbose=False)[0]) for k, p in proteins.items()}
vmax = max(m.max() for m in maps.values())
extent = [1, 140, 140, 1]    # residue numbers

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.1), constrained_layout=True)
for ax, (k, m) in zip(axes, maps.items()):
    im = ax.imshow(m, cmap="viridis", vmin=0, vmax=vmax, extent=extent)
    ax.set_xlabel("Residue number")
    ax.text(0.03, 0.03, k, transform=ax.transAxes, color="white", fontsize=9)
axes[0].set_ylabel("Residue number")
cbar = fig.colorbar(im, ax=axes, shrink=0.85)
cbar.set_label("Mean Cα-Cα distance (Å)")
plt.show()

## Where do they differ?

The difference map, IDPForge minus AF-CALVADOS, makes the comparison direct: red where IDPForge's residue pairs are further apart than AF-CALVADOS's, blue where they are closer.

In [ ]:
diff = maps["IDPForge"] - maps["AF-CALVADOS"]
limit = np.abs(diff).max()

fig, ax = plt.subplots(figsize=(3.9, 3.2))
im = ax.imshow(diff, cmap="RdBu_r", vmin=-limit, vmax=limit, extent=extent)
ax.set_xlabel("Residue number")
ax.set_ylabel("Residue number")
cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label("Δ distance (Å)")
plt.tight_layout()
plt.show()
print(f"Largest differences: {diff.min():.1f} to {diff.max():.1f} A")

## Deviation from a homopolymer

`get_polymer_scaled_distance_map()` compares each RMS distance with a homopolymer of the same scaling exponent (by default fitted to the same ensemble). This removes the trivial dependence on sequence separation and shows which regions are more compact (blue) or more expanded (red) than an equivalent uniform chain would be.

In [ ]:
scaled, nu, A0, redchi = proteins["IDPForge"].get_polymer_scaled_distance_map(
    mode="signed-fractional-change", verbose=False
)
print(f"Homopolymer reference: nu = {nu:.3f}, A0 = {A0:.2f} A")
scaled = np.where(scaled == 0, np.nan, scaled)     # blank the excluded short-range pairs
limit = np.nanmax(np.abs(scaled))

fig, ax = plt.subplots(figsize=(3.9, 3.2))
im = ax.imshow(scaled, cmap="RdBu_r", vmin=-limit, vmax=limit, extent=extent)
ax.set_xlabel("Residue number")
ax.set_ylabel("Residue number")
cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label("Fractional deviation from homopolymer")
plt.tight_layout()
plt.show()

## Contacts

`get_contact_map()` gives the fraction of conformers in which each residue pair is in contact (here, any two heavy atoms within 5 Å). It also returns a per-residue contact order: how often each residue contacts residues at least three positions away.

Contacts in disordered ensembles are dominated by residues close in sequence, so the colour scale is capped at 0.1. That makes the rarer long-range contacts visible.

In [ ]:
contacts = {k: p.get_contact_map(distance_thresh=5.0, mode="closest-heavy") for k, p in proteins.items()}

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.1), constrained_layout=True)
for ax, (k, (cmap, order)) in zip(axes, contacts.items()):
    im = ax.imshow(cmap, cmap="Greys", vmin=0, vmax=0.1, extent=extent)
    ax.set_xlabel("Residue number")
    ax.text(0.03, 0.03, k, transform=ax.transAxes, fontsize=9)
axes[0].set_ylabel("Residue number")
cbar = fig.colorbar(im, ax=axes, shrink=0.85, extend="max")
cbar.set_label("Contact fraction")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4.4, 2.6))
for k, (cmap, order) in contacts.items():
    ax.plot(np.arange(1, len(order) + 1), order, lw=1, label=k)
ax.set_xlabel("Residue number")
ax.set_ylabel("Contact order")
ax.set_xlim(1, 140)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()